<h1 style="color:yellow;font-size:xxx-large">
Setup
</h1>

In [ ]:
import ansys.fluent.core as pyfluent
%matplotlib notebook
import numpy as np
import os
import math as math
import time as time
import warnings

In [ ]:
# Data Setup
respiratory_rate = 13.2 #bpm
size_of_timestep = 0.0001 #sec
iterations_per_timestep = 50
number_of_respirations = 1
working_directory = os.getcwd()
new_udf_file_name = "inlet_velocity.c"
mesh_file_name = "new-model-only-decimated-1-06skew-032orth-481Mcells.msh.h5"

In [ ]:
# Data Calculation
respiratory_period = 60/respiratory_rate #sec
inhalation_time = 20/respiratory_rate #sec
exhalation_time = 40/respiratory_rate #sec
omega_in = respiratory_rate*np.pi/20 #1/sec
omega_out = respiratory_rate*np.pi/40 #1/sec
peak_inhalation_rate = 27*respiratory_rate*np.pi/40 #L/min
peak_exhalation_rate = 27*respiratory_rate*np.pi/80 #L/min

#starting the particle injection in the beginning of last respiratory cycle
particle_injection_start_time = (number_of_respirations-1)*respiratory_period #sec
#ending the particle inhjection during the last cycle's exhalation start time
particle_injection_end_time = number_of_respirations*respiratory_period-exhalation_time #sec

number_of_timesteps = 3

<h1 style="color:yellow;font-size:xxx-large">
Start Solver
</h1>

In [ ]:
#launch fluent in solver
solver = pyfluent.launch_fluent(precision='double', 
                                processor_count=12, 
                                ui_mode='gui', 
                                mode='solver', 
                                start_transcript=False)
warnings.filterwarnings("ignore")

# Read Steady Case and Replace Mesh

In [ ]:
solver.settings.file.read_case(file_name="steady_case.cas.h5")
solver.settings.file.replace_mesh(file_name=str("Meshes\\" + mesh_file_name))

# Set convergence criteria

In [ ]:
equations = solver.solution.monitor.residual.equations
equations["continuity"].absolute_criteria = 0.001
equations["k"].absolute_criteria = 0.001
equations["omega"].absolute_criteria = 0.001
equations["x-velocity"].absolute_criteria = 0.001
equations["y-velocity"].absolute_criteria = 0.001
equations["z-velocity"].absolute_criteria = 0.001

# Calculate peak inhalation and exhalation velocities

In [ ]:
from peak_velocity_calc import find_peak_velo
peak_inhalation_velocity = find_peak_velo(solver_session = solver,
                                          working_directory=working_directory,
                                          peak_flow_rate = peak_inhalation_rate,
                                          calc_area = "nostrils",
                                          area_file_name = "surface_integral",
                                          unit = "L/min")

In [ ]:
print(peak_inhalation_velocity)

2.5932689071389787


In [ ]:
peak_exhalation_velocity = find_peak_velo(solver_session=solver,
                                          working_directory=working_directory,
                                          peak_flow_rate=peak_exhalation_rate,
                                          calc_area="nostrils",
                                          area_file_name="surface_integral",
                                          unit="L/min")

In [ ]:
print(peak_exhalation_velocity)

1.2966344535694894


# Assign peak inhalation velocity to nostrils' inlet

In [ ]:
solver.settings.setup.boundary_conditions.velocity_inlet["nostrils"].momentum.velocity_magnitude.value = peak_inhalation_velocity

<h1 style="color:yellow;font-size:xxx-large">
Initialization
</h1>

In [ ]:
# initialization
initialization = solver.settings.solution.initialization
initialization.initialization_type = "hybrid"
initialization.initialize()

# Start Steady State Calculations

In [ ]:
calc_para = solver.settings.solution.run_calculation.parameters
calc_para.iter_count = 1000
calc_para.reporting_interval = 5
solver.settings.solution.run_calculation.calculate()

# Write case and data files

In [ ]:
path_for_case_data = str("D:\\Lii\\Results\\simulation_results_steady")
solver.tui.file.write_case_data(path_for_case_data)

# Set Transient General Settings

In [ ]:
from general_setting_setup import general_settings_transients
general_settings_transients(solver_session=solver)
solver.settings.solution.methods.p_v_coupling.flow_scheme = "SIMPLE"

# Write the UDF file

In [ ]:
# info to include regarding the number of breathing cycles
t = 0
code_text = ""
i = 1
for i in range(1, number_of_respirations+1):
    if i == 1:
        code_text = code_text + "\n /*inhalation no " + str(i) + "*/" + "\nif (t<" + str(inhalation_time) + ")\n{\nvelocity = AIN*sin(OMEGAIN*t);\n}" + "\n/*exhalation no " + str(i) + "*/" +"\nelse if(t>=" + str(inhalation_time) + " && t<" + str(respiratory_period) +")\n{\nvelocity = -AOUT*sin(OMEGAOUT*(t-" + str(inhalation_time) +"));\n}"
    else:
        #describing number of inhalations
        code_text = code_text + "\n/*inhalation no " + str(i) + "*/"
        #adding inhalation else if clause
        code_text = code_text + "\nelse if(t>=" + str(i-1) + "*" + str(respiratory_period) + " && " + "t<" + str((i-1)*respiratory_period+inhalation_time)
        #adding inhalation calculation code
        code_text = code_text + "\n{" + "\nvelocity = AIN*sin(OMEGAIN*(t-(" + str((i-1)*respiratory_period) + "));" + "\n}"
        #describing number of exhalation
        code_text = code_text + "\n/*exhalation no " + str(i) + "*/"
        #adding exhalation else if clause
        code_text = code_text + "\nelse if(t>=" + str((i-1)*respiratory_period+inhalation_time) + " && " + "t<" + str(i*respiratory_period)
        #adding exhalation calculation code
        code_text = code_text + "\n{" + "\nvelocity = -AOUT*sin(OMEGAOUT*(t-" + str((i-1)*respiratory_period + inhalation_time) +"));" + "\n}"

In [ ]:
# list of parts needed to change in the file
changable_text = ["double AIN = 0;", 
                  "double AOUT = 0;", 
                  "double OMEGAIN = 0;", 
                  "double OMEGAOUT = 0;",
                  "code_text"]
new_text = ["double AIN = " + str(peak_inhalation_velocity) + ";",
            "double AOUT = " + str(peak_exhalation_velocity) + ";",
            "double OMEGAIN = " + str(omega_in) + ";",
            "double OMEGAOUT = " + str(omega_out) + ";",
            code_text]



In [ ]:
from change_udf_file import change_udf_file
change_udf_file(working_directory=working_directory,
                udf_file_name="changing_inlet_velocity.txt",
                list_of_changed_text=changable_text,
                list_of_new_texts=new_text,
                new_file_name=new_udf_file_name)

<h1 style="color:yellow;font-size:xxx-large">
UDF interpretation
</h1>

In [ ]:
help(solver.setup.user_defined.interpreted_functions(file_name="inlet_velocity.c"))

Help on NoneType object:

class NoneType(object)
 |  The type of the None singleton.
 |
 |  Methods defined here:
 |
 |  __bool__(self, /)
 |      True if self else False
 |
 |  __eq__(self, value, /)
 |      Return self==value.
 |
 |  __ge__(self, value, /)
 |      Return self>=value.
 |
 |  __gt__(self, value, /)
 |      Return self>value.
 |
 |  __hash__(self, /)
 |      Return hash(self).
 |
 |  __le__(self, value, /)
 |      Return self<=value.
 |
 |  __lt__(self, value, /)
 |      Return self<value.
 |
 |  __ne__(self, value, /)
 |      Return self!=value.
 |
 |  __repr__(self, /)
 |      Return repr(self).
 |
 |  ----------------------------------------------------------------------
 |  Static methods defined here:
 |
 |  __new__(*args, **kwargs)
 |      Create and return a new object.  See help(type) for accurate signature.



# Assign UDF to velocity inlet

In [ ]:
# assign the udf function to inlet velocity
solver.settings.setup.boundary_conditions.velocity_inlet["nostrils"].momentum.velocity.option = "udf"

# Load tabulated data to Fluent for particle injection

In [ ]:
solver.file.table_file_manager.read_table_file(file_name="baseline_30_particles.txt")

# Assign DPM Settings

In [ ]:
# discrete phase settings
discrete_phase = solver.settings.setup.models.discrete_phase
discrete_phase.general_settings.unsteady_tracking.enabled = True
discrete_phase.tracking.max_num_steps = 5000
discrete_phase.physical_models.erosion_accretion_enabled = True
discrete_phase.tracking.step_size_controls

# Create Particle Injection

In [ ]:
from part_inject import create_part_inject
create_part_inject(solver_session=solver, 
                   injection_name="part-inject", 
                   injection_start_time=particle_injection_start_time, 
                   injection_stop_time=particle_injection_end_time,
                   injection_surface="nostrils",
                   table_name="baseline_30_rate",
                   column_with_diameters="drop-diam",
                   column_with_number_fractions="num-frac",)

# Assign wall boundary condition to "trap"

In [ ]:
solver.settings.setup.boundary_conditions.wall["new-model-only-decimated-1-facetten"].dpm.discrete_phase_bc_type = "trap"

# Create inlet flow report definition

In [ ]:
report_def = solver.settings.solution.report_definitions
report_def.surface.create("inlet-flow")
inlet_flow_definition = report_def["inlet-flow"]
inlet_flow_definition.surface_names = "nostrils"
inlet_flow_definition.report_type = "surface-areaavg"
inlet_flow_definition.field = "velocity-magnitude"

# create inlet flow velocity magnitude report plot

In [ ]:
report_plots = solver.settings.solution.monitor.report_plots
report_plots.create("inlet-flow-report")
inlet_flow_report = report_plots["inlet-flow-report"]
inlet_flow_report.plot_window = 2
inlet_flow_report.report_defs = "inlet-flow"
inlet_flow_report.print = True
inlet_flow_report.plot_instantaneous_values = True

# Assign autosave settings

In [ ]:
auto_save = solver.settings.solution.calculation_activity.auto_save
auto_save.case_frequency = "if-case-is-modified"
auto_save.data_frequency = 1000
auto_save.root_name = "./simulation_autosave"
auto_save.append_file_name_with.file_suffix_type = "flow-time"
auto_save.save_data_file_every.frequency_type = "time-step"

In [ ]:
solver.settings.file.read_journal(file_name_list=["start-transient-dpm-sample.jou"])

In [ ]:
transient_controls = solver.settings.solution.run_calculation.transient_controls
transient_controls.time_step_count = number_of_timesteps
transient_controls.time_step_size = size_of_timestep
transient_controls.max_iter_per_time_step = iterations_per_timestep
solver.settings.solution.run_calculation.calculate()

In [ ]:
solver.settings.file.read_journal(file_name_list=["stop-transient-dpm-sample.jou"])

In [ ]:
#save case_data file to the working directory
solver.tui.file.write_case_data("simulation-results-test")

The following solver settings object method could also be used to execute the above command:
<solver_session>.settings.file.write(file_type = 'case-data', file_name = 'simulation-results-test')


In [ ]:
#solver.exit()